In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/illumShift/datasetTreesIllumShift-shadowClas.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,shadowInd,grviOut
0,0.224674,0.000277,0.000397,1141.969,1031.228,0.0,180.0,-63.812729,-56.07412,0.0,-19.35,-0.0,0,0.088270
1,0.196116,0.000391,0.000550,1141.969,1031.228,1.0,180.0,-63.812729,-56.07412,0.0,-19.35,-0.0,0,0.057202
2,0.202034,0.000388,0.000555,1141.969,1031.228,2.0,180.0,-63.812729,-56.07412,0.0,-19.35,-0.0,0,0.048503
3,0.185897,0.000346,0.000477,1141.969,1031.228,3.0,180.0,-63.812729,-56.07412,0.0,-19.35,-0.0,0,0.086477
4,0.233005,0.000312,0.000456,1141.969,1031.228,4.0,180.0,-63.812729,-56.07412,0.0,-19.35,-0.0,0,0.067095


In [4]:
target_col = dataset["shadowInd"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0., 0., 0., ..., 0., 0., 0.], shape=(332640,), dtype=float32)

In [5]:
inputs = dataset.drop(["grviOut", "shadowInd"], axis=1)
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.2467394e-01,  2.7675237e-04,  3.9742354e-04, ...,
         0.0000000e+00, -1.9350000e+01, -0.0000000e+00],
       [ 1.9611603e-01,  3.9089142e-04,  5.4955034e-04, ...,
         0.0000000e+00, -1.9350000e+01, -0.0000000e+00],
       [ 2.0203422e-01,  3.8837871e-04,  5.5490516e-04, ...,
         0.0000000e+00, -1.9350000e+01, -0.0000000e+00],
       ...,
       [ 2.8367174e-01,  2.6904649e-04,  4.2221454e-04, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02],
       [ 2.4132200e-01,  3.4478656e-04,  5.0772808e-04, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02],
       [ 2.7499148e-01,  3.2630350e-04,  5.1217148e-04, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02]],
      shape=(332640, 12), dtype=float32)

In [6]:
if len(labels) != len (inputs):
    print ("problem")

In [7]:
n_total = len(inputs)

n_train = int(0.8*n_total)
n_test = int(0.1*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]
test_inputs, test_labels = inputs_shuf[n_train + n_val:], labels_shuf[n_train + n_val:]

In [8]:
# scaling
from sklearn.preprocessing import QuantileTransformer, StandardScaler

if True:
    input_scaler = StandardScaler() #QuantileTransformer(output_distribution='normal')
    target_scaler = StandardScaler()

    train_inputs = input_scaler.fit_transform(train_inputs)
    #train_labels = target_scaler.fit_transform(train_labels.reshape(-1, 1)).reshape(-1)

    val_inputs = input_scaler.transform(val_inputs)
    #val_labels = target_scaler.transform(val_labels.reshape(-1, 1)).reshape(-1)

    test_inputs = input_scaler.transform(test_inputs)
    #test_labels = target_scaler.transform(test_labels.reshape(-1, 1)).reshape(-1)

In [9]:
train_inputs = torch.from_numpy(train_inputs).float()
train_labels = torch.from_numpy(train_labels).float()

val_inputs = torch.from_numpy(val_inputs).float()
val_labels = torch.from_numpy(val_labels).float()

test_inputs = torch.from_numpy(test_inputs).float()
test_labels = torch.from_numpy(test_labels).float()

In [10]:
print ("Train size:", len(train_inputs))
print ("Val size:", len(val_inputs))
print ("Test size:", len(test_inputs))

Train size: 266112
Val size: 33264
Test size: 33264


In [11]:
BATCH_SIZE = 512

class metricsDataset (Dataset):
    def __init__(self, metrics, labels):
        self.metrics = metrics
        self.labels = labels

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], self.labels[idx]

train_dataset = metricsDataset(train_inputs, train_labels)
val_dataset   = metricsDataset(val_inputs, val_labels)
test_dataset  = metricsDataset(test_inputs, test_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [13]:
model = nn.Sequential(
    # nn.Linear(12, 128),
    # nn.BatchNorm1d(128),
    # nn.ReLU(),

    nn.Linear(12, 16),
    nn.ReLU(),

    nn.Linear(16, 1)
).to(device)

In [1]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

NameError: name 'nn' is not defined

In [15]:
EPOCHS = 160

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        optimizer.zero_grad()

        logits = model(x).squeeze(-1)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()

            logits = model(x).squeeze(-1)

            loss = criterion(logits, y)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"Train BCE: {train_loss:.6f} | "
        f"Val BCE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        }, "best_model_checkpoint.pth")

        print("Best Model Saved")

Epoch   1/160 | Train BCE: 0.280404 | Val BCE: 0.088045
Best Model Saved
Epoch   2/160 | Train BCE: 0.046669 | Val BCE: 0.026106
Best Model Saved
Epoch   3/160 | Train BCE: 0.017433 | Val BCE: 0.013425
Best Model Saved
Epoch   4/160 | Train BCE: 0.009521 | Val BCE: 0.008505
Best Model Saved
Epoch   5/160 | Train BCE: 0.006070 | Val BCE: 0.006077
Best Model Saved
Epoch   6/160 | Train BCE: 0.004290 | Val BCE: 0.004594
Best Model Saved
Epoch   7/160 | Train BCE: 0.003217 | Val BCE: 0.003738
Best Model Saved
Epoch   8/160 | Train BCE: 0.002553 | Val BCE: 0.003103
Best Model Saved
Epoch   9/160 | Train BCE: 0.002082 | Val BCE: 0.002683
Best Model Saved
Epoch  10/160 | Train BCE: 0.001749 | Val BCE: 0.002343
Best Model Saved
Epoch  11/160 | Train BCE: 0.001542 | Val BCE: 0.002156
Best Model Saved
Epoch  12/160 | Train BCE: 0.001358 | Val BCE: 0.001950
Best Model Saved
Epoch  13/160 | Train BCE: 0.001211 | Val BCE: 0.001765
Best Model Saved
Epoch  14/160 | Train BCE: 0.001127 | Val BCE: 0.00

In [16]:
checkpoint = torch.load(
    "best_model_checkpoint.pth",
    map_location=device
)
model.load_state_dict(checkpoint["model_state_dict"])
print(
    f"Loaded best model "
    f"(Val BCE: {checkpoint['best_val_loss']:.6f})"
)

model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        logits = model(x).squeeze(-1)

        loss = criterion(logits, y)
        test_loss += loss.item() * x.size(0)

        probabilities = torch.sigmoid(logits)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == y).sum().item()
        total += y.numel()

test_loss /= total
accuracy = 100.0 * correct / total

print(f"\nTest BCE Loss: {test_loss:.6f}")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Correct: {correct}/{total}")

Loaded best model (Val BCE: 0.000852)

Test BCE Loss: 0.000145
Test Accuracy: 100.00%
Correct: 33264/33264
